In [1]:
import os
import pickle
from collections import Counter
from typing import Dict, List, Tuple, FrozenSet

In [2]:
DIR = "../case_study/T_cell/N10/results/20260615/"

In [3]:
PI = Dict[str, int]
NodePIs = List[List[PI]]  # [inactivation, activation]
Networks = List[Dict[str, NodePIs]]

In [4]:
def canon_pi(pi: PI) -> FrozenSet[Tuple[str, int]]:
    return frozenset(pi.items())


def canon_node(node_pis: NodePIs):
    """
    Full node rule:
    (inactivation set, activation set)
    """
    inact = frozenset(canon_pi(pi) for pi in node_pis[0])
    act = frozenset(canon_pi(pi) for pi in node_pis[1])
    return (inact, act)

In [5]:
def split_nodes(networks: Networks):
    nodes = networks[0].keys()

    node_versions = {node: [] for node in nodes}

    for net in networks:
        for node in nodes:
            node_versions[node].append(canon_node(net[node]))

    invariant_nodes = {}
    variable_nodes = {}

    for node, versions in node_versions.items():
        first = versions[0]

        if all(v == first for v in versions):
            invariant_nodes[node] = first
        else:
            variable_nodes[node] = versions

    return invariant_nodes, variable_nodes

In [6]:
def consensus_from_variable_nodes(variable_nodes, threshold: float, n_networks: int):
    result = {}

    for node, versions in variable_nodes.items():
        inact_counter = Counter()
        act_counter = Counter()

        for (inact_set, act_set) in versions:
            inact_counter.update(inact_set)
            act_counter.update(act_set)

        def filter(counter: Counter):
            return [
                dict(pi)
                for pi, count in counter.items()
                if (count / n_networks) >= threshold
            ]

        result[node] = [
            filter(inact_counter),
            filter(act_counter),
        ]

    return result

In [7]:
def build_consensus(networks: Networks, threshold: float = 1.0):
    n_networks = len(networks)

    invariant, variable = split_nodes(networks)

    consensus = consensus_from_variable_nodes(variable, threshold, n_networks)

    return {
        "consensus": consensus,
        "invariant_nodes": invariant
    }

In [8]:
# primes1 = pickle.load(open("../case_study/T_cell/Tcell_primes.pkl", "rb"))
# primes2 = pickle.load(open("../case_study/T_cell/generated_models/20260613_1/Tcell_4405_gen28_primes.pkl", "rb"))
# networks = [primes1, primes2]

In [9]:
# read all pickle files in a target directory
directory = os.listdir(DIR)


networks = []
for filename in directory:
    if filename.endswith(".pkl"):
        primes = pickle.load(open(os.path.join(DIR, filename), "rb"))
        networks.append(primes)

print(len(networks))

68


In [10]:
consensus = build_consensus(networks)
consensus

{'consensus': {'FOXP3': [[], []],
  'GATA3': [[{'TBET': 1}], [{'GATA3': 1, 'TBET': 0}]],
  'IFNG': [[{'STAT4': 0, 'FOXP3': 1},
    {'STAT4': 0, 'TBET': 0},
    {'STAT4': 0, 'NFAT': 0},
    {'STAT4': 0, 'RUNX3': 0}],
   [{'STAT3': 0,
     'FOXP3': 0,
     'proliferation': 1,
     'TBET': 1,
     'NFAT': 1,
     'RUNX3': 1}]],
  'IL17': [[], []],
  'IL2': [[], []],
  'IL23R': [[{'STAT3': 0}, {'IL23': 0, 'IL23_e': 0}],
   [{'RORGT': 1, 'IL23_e': 1, 'STAT3': 1}]],
  'IL2R': [[], []],
  'IL2RA': [[{'NFKB': 0, 'NFAT': 0, 'SMAD3': 0}], [{'NFAT': 1, 'STAT5': 1}]],
  'IL4': [[], []],
  'IL4R': [[], []],
  'IL4R_2': [[{'IL4R': 0}], [{'IL4RA_2': 1, 'IL4_e': 1, 'IL4R': 1}]],
  'STAT3': [[], [{'IL21R': 1}, {'IL6R': 1}, {'IL27R': 1}]],
  'STAT5': [[{'IL2R': 0, 'IL15R': 0, 'IL4R': 0, 'STAT5_2': 0}],
   [{'STAT5_2': 1, 'STAT5': 1}, {'IL4R': 1}]],
  'STAT5_2': [[{'STAT5': 0}], []],
  'TBET': [[{'STAT1': 0, 'TBET': 0}], [{'GATA3': 0, 'STAT1': 1}]],
  'TGFB': [[{'NFAT': 0}], []],
  'proliferation': [[{'p

In [11]:
invariant_nodes = list(consensus["invariant_nodes"].keys())
variable_nodes = list(consensus["consensus"].keys())

print(f"{len(invariant_nodes)} invariant nodes:", list(consensus["invariant_nodes"].keys()))
print(f"{len(variable_nodes)} variable nodes:", list(consensus["consensus"].keys()))

41 invariant nodes: ['APC', 'CD28', 'IFNBR', 'IFNB_e', 'IFNGR', 'IFNG_e', 'IKB', 'IL10', 'IL10R', 'IL10_e', 'IL12R', 'IL12RB1_2', 'IL12RB2', 'IL12_e', 'IL15R', 'IL15_e', 'IL21', 'IL21R', 'IL21_e', 'IL23', 'IL23_e', 'IL27R', 'IL27_e', 'IL2R_2', 'IL2_e', 'IL4RA_2', 'IL4_e', 'IL6R', 'IL6_e', 'IRF1', 'NFAT', 'NFKB', 'RORGT', 'RUNX3', 'SMAD3', 'STAT1', 'STAT4', 'STAT6', 'TCR', 'TGFBR', 'TGFB_e']
17 variable nodes: ['FOXP3', 'GATA3', 'IFNG', 'IL17', 'IL2', 'IL23R', 'IL2R', 'IL2RA', 'IL4', 'IL4R', 'IL4R_2', 'STAT3', 'STAT5', 'STAT5_2', 'TBET', 'TGFB', 'proliferation']
